In [1]:
#import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
#from itertools import product
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import seaborn as sns
import warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

## where trained models are saved (project-root models/ folder)
MODEL_DIR = Path("../models").resolve()
MODEL_DIR.mkdir(exist_ok=True)

In [2]:
df_2024 = pd.read_csv("../data/df_2024_final.csv").assign(season=2024)
df_2025 = pd.read_csv("../data/df_2025_final.csv").assign(season=2025)
df_2026 = pd.read_csv("../data/df_2026_final.csv").assign(season=2026)

# make 2026 consistent with 2024/2025: drop the 2026-only columns and use the same column order
# (2026 lacks the defensive columns, which are ~all NaN in 2025 and unused -> filled with NaN)
df_2026 = df_2026.drop(columns=["pro_team_id", "season_type"], errors="ignore")
df_2026 = df_2026.reindex(columns=df_2024.columns)
df_2026 = df_2026.astype(df_2024.dtypes.to_dict(), errors="ignore")

combined = pd.concat([df_2024, df_2025, df_2026], ignore_index=True)

In [3]:
#Sort chronologically by player
combined = combined.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)

#Create target (Next week's fantasy score) and next week's context via shift(-1)
#FIX: group by player AND season so a season's last game doesn't take next season's week 1 as its target
g = combined.groupby(['player_id', 'season'])
combined['target_next_week_fantasy'] = g['fantasy_points'].shift(-1)
combined['next_week_opponent'] = g['opponent'].shift(-1)
combined['next_week_atVs'] = g['atVs'].shift(-1)
combined['next_week_projected'] = g['projected_points'].shift(-1)   # ESPN's projection for the game being predicted
for col in ['roof', 'surface', 'temp', 'wind', 'location']:
    combined[f'next_week_{col}'] = g[col].shift(-1)
# next_week_opp_def_rank removed: opp_def_rank is a season-end value, so it leaks future games

In [4]:
#Compute cumulative career and season-to-date stats up to current row (Week N)
stats_to_accumulate = [
    'fantasy_points', 'receptions', 'receivingTargets', 'receivingYards', 
    'receivingTouchdowns', 'rushingAttempts', 'rushingYards', 'rushingTouchdowns',
    'completions', 'passingAttempts', 'passingYards', 'passingTouchdowns', 
    'interceptions', 'fumblesLost'
]

for col in stats_to_accumulate:
    # 1. Career cumulative mean (Extract values to bypass MultiIndex mismatch)
    combined[f'{col}_cum_career_avg'] = (
        combined.groupby('player_id')[col]
        .expanding()
        .mean()
        .values
    )
    
    # 2. Season-to-date cumulative mean
    combined[f'{col}_season_avg'] = (
        combined.groupby(['player_id', 'season'])[col]
        .expanding()
        .mean()
        .values
    )

for col in ['fantasy_points', 'receivingTargets', 'rushingAttempts', 'passingAttempts']:
    combined[f'{col}_last3'] = (
        combined.groupby(['player_id', 'season'])[col]
        .transform(lambda s: s.rolling(3, min_periods=1).mean())
    )

In [5]:
# combined = combined[(combined['season'] == 2024) | (combined['season'] == 2025) & (combined['week'] < 18)]

In [6]:
categorical_cols = ['player','position', 'next_week_opponent', 'next_week_atVs', 'next_week_surface',
                    'next_week_roof', 'next_week_location']
for col in categorical_cols:
    if col in combined.columns:
        combined[col] = combined[col].astype('category')

In [7]:
irrelevant_columns = [
    'Unnamed: 0', 'player_id', 'team', 'opponent', 'date',
    'eventId', 'game_id', 'gameday', 'gametime', 'stadium', 'result',
    'score', 'fantasy_points', 'target_next_week_fantasy', 'espn',
    '_set', 'season',
    'med_proj_pts',
    'projected_points',
    'atVs', 'roof', 'surface', 'temp', 'wind', 'location',
    'opp_def_rank', 'team_pass_rank', 'team_rush_rank', 'next_week_projected' 
]

relevant_columns = [col for col in combined.columns if col not in irrelevant_columns]

In [8]:
relevant_columns

['player',
 'position',
 'week',
 'receptions',
 'receivingTargets',
 'receivingYards',
 'yardsPerReception',
 'receivingTouchdowns',
 'longReception',
 'rushingAttempts',
 'rushingYards',
 'yardsPerRushAttempt',
 'longRushing',
 'rushingTouchdowns',
 'fumbles',
 'fumblesLost',
 'fumblesForced',
 'kicksBlocked',
 'completions',
 'passingAttempts',
 'passingYards',
 'completionPct',
 'yardsPerPassAttempt',
 'passingTouchdowns',
 'interceptions',
 'longPassing',
 'sacks',
 'QBRating',
 'adjQBR',
 'longFieldGoalMade',
 'fieldGoalPct',
 'fieldGoalsMadeYardsAverage',
 'totalKickingPoints',
 'totalTackles',
 'soloTackles',
 'assistTackles',
 'stuffs',
 'stuffYards',
 'fumblesRecovered',
 'interceptionYards',
 'avgInterceptionYards',
 'interceptionTouchdowns',
 'longInterception',
 'passesDefended',
 'fieldGoalsMade1_19',
 'fieldGoalAttempts1_19',
 'fieldGoalsMade20_29',
 'fieldGoalAttempts20_29',
 'fieldGoalsMade30_39',
 'fieldGoalAttempts30_39',
 'fieldGoalsMade40_49',
 'fieldGoalAttempts40

In [9]:
target = 'target_next_week_fantasy'

In [10]:
train_data = combined[
    (combined['season'] == 2024) |
    ((combined['season'] == 2025) & (combined['week'] < 10))
]
val_data = combined[
    (combined['season'] == 2025) & (combined['week'].between(10, 18))
]
test_data = combined[
    (combined['season'] == 2026) & (combined['week'] < 11)
]
print(len(train_data), len(val_data), len(test_data))

# rebuild from scratch so stale cells can't interfere
X_train, y_train = train_data[relevant_columns], train_data[target]
X_val,   y_val   = val_data[relevant_columns],   val_data[target]
X_test,  y_test  = test_data[relevant_columns],  test_data[target]

# diagnostic: how many bad labels are there?
for name, y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(name, 'NaN:', y.isna().sum(), 'inf:', np.isinf(y).sum(), 'rows:', len(y))

# keep only rows with a finite target
def clean(X, y):
    m = y.notna() & np.isfinite(y)
    return X[m], y[m]

X_train, y_train = clean(X_train, y_train)
X_val,   y_val   = clean(X_val,   y_val)
X_test,  y_test  = clean(X_test,  y_test)

print(len(X_train), len(X_val), len(X_test))

11286 3808 4440
train NaN: 731 inf: 0 rows: 11286
val NaN: 625 inf: 0 rows: 3808
test NaN: 1190 inf: 0 rows: 4440
10555 3183 3250


In [11]:
model_path = MODEL_DIR / "xgb_global.json"   # where this model is saved after training
model = xgb.XGBRegressor(
    n_estimators=10000, #Higher is better
    learning_rate=0.005, #lower is better
    max_depth=5, #higher is better
    min_child_weight=3, #mid:1-10
    subsample=0.7, #mid:0-1
    colsample_bytree=0.7, #mid:0-1
    random_state=12, #seed
    enable_categorical=True,
    early_stopping_rounds=50      # stops when validation stops improving (old run overfit after ~50 trees)
)

In [12]:
# model = xgb.XGBRegressor(
#     n_estimators=10000, #Higher is better
#     learning_rate=0.01, #lower is better
#     max_depth=8, #higher is better
#     min_child_weight=5, #mid:1-10
#     subsample=0.5, #mid:0-1
#     colsample_bytree=0.5, #mid:0-1
#     random_state=42, #seed
#     enable_categorical=True,
#     early_stopping_rounds=50      # stops when validation stops improving (old run overfit after ~50 trees)
# ) 806

In [13]:
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)
model.save_model(str(model_path))
print(f"saved model to {model_path}")

## reload later without retraining:
# model = xgb.XGBRegressor()
# model.load_model(str(model_path))

[0]	validation_0-rmse:7.58224
[50]	validation_0-rmse:6.90665
[100]	validation_0-rmse:6.47621
[150]	validation_0-rmse:6.21190
[200]	validation_0-rmse:6.05913
[250]	validation_0-rmse:5.97167
[300]	validation_0-rmse:5.93139
[350]	validation_0-rmse:5.91360
[400]	validation_0-rmse:5.91001
[449]	validation_0-rmse:5.91708
saved model to /home/tristan/Documents/MSDS/CSCI 5502/CSCI_5502_Fantasy_Project/models/xgb_global.json


In [14]:
test_results = combined.loc[X_test.index].copy()
test_results['model_predicted_score'] = model.predict(X_test)
test_results['actual_score'] = y_test                      # next week's actual
test_results['espn_projection'] = test_results['next_week_projected']  # ESPN's projection for that same game

test_results['model_error'] = test_results['actual_score'] - test_results['model_predicted_score']
test_results['model_abs_error'] = test_results['model_error'].abs()
test_results['espn_error'] = test_results['actual_score'] - test_results['espn_projection']
test_results['espn_abs_error'] = test_results['espn_error'].abs()

# only compare rows where ESPN has a projection
cmp = test_results.dropna(subset=['espn_projection']).copy()

cmp['winner'] = np.select(
    [cmp['model_abs_error'] < cmp['espn_abs_error'],
     cmp['model_abs_error'] > cmp['espn_abs_error']],
    ['Model', 'ESPN'],
    default='Tie'
)

# overall metrics
for name, pred in [('Model', cmp['model_predicted_score']), ('ESPN', cmp['espn_projection'])]:
    print(f"{name}: MAE={mean_absolute_error(cmp['actual_score'], pred):.3f}  "
          f"RMSE={np.sqrt(mean_squared_error(cmp['actual_score'], pred)):.3f}  "
          f"R2={r2_score(cmp['actual_score'], pred):.3f}")

# biggest model misses
big_misses = cmp[cmp['model_abs_error'] > 2].sort_values('week', ascending=False)#.sort_values('model_abs_error', ascending=False)
big_misses[['season', 'week', 'player', 'position', 'team', 'actual_score',
            'model_predicted_score', 'espn_projection', 'model_error', 'espn_error', 'winner']].head(10)

Model: MAE=4.376  RMSE=5.947  R2=0.412
ESPN: MAE=3.868  RMSE=5.504  R2=0.496


,season,week,player,position,team,actual_score,model_predicted_score,espn_projection,model_error,espn_error,winner
19339,2026,8,Harold Fannin Jr.,TE,CLE,12.20,9.951295,11.523310,2.248705,0.676690,ESPN
219,2026,8,Matthew Stafford,QB,LAR,11.22,15.567139,16.907602,-4.347139,-5.687602,Model
19414,2026,8,Andrew Armstrong,WR,KC,0.00,2.462993,1.084953,-2.462993,-1.084953,ESPN
19528,2026,8,Ted Hurst III,WR,TB,0.00,3.915630,5.669760,-3.915630,-5.669760,Model
19409,2026,8,Colbie Young,WR,CIN,0.00,2.111796,1.333871,-2.111796,-1.333871,ESPN
18684,2026,8,Omar Cooper Jr.,WR,NYJ,13.80,7.223398,11.108534,6.576602,2.691466,ESPN
18692,2026,8,Camden Brown,WR,DAL,0.00,2.103664,1.685620,-2.103664,-1.685620,ESPN
18713,2026,8,Mason Taylor,TE,NYJ,2.10,7.627793,8.968149,-5.527793,-6.868149,Model
6372,2026,8,Jordan Love,QB,GB,36.18,15.408814,17.030534,20.771186,19.149466,ESPN
5418,2026,8,Saquon Barkley,RB,PHI,8.20,17.756821,12.457859,-9.556821,-4.257859,ESPN


In [15]:
position_table = pd.crosstab(cmp['position'], cmp['winner'], margins=True)
print(position_table)

winner    ESPN  Model   All
position                   
K          109    127   236
QB         180    106   286
RB         433    307   740
TE         493    257   750
WR         676    475  1151
All       1891   1272  3163
